In [2]:
## 0. Config & imports
import os
from pathlib import Path
import datetime
import json
import random

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt  # for optional plots

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---- Identity / tags ----
SCRIPT_NAME = "220_autoencoder_ersp_matrix.py"
ALGO_TAG = "autoencoder_clean"
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")  # e.g. 20251117_112233

# ---- Paths ----
ANALYSIS_ROOT = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora")

# Input: RAWONLY ERSP matrices (numeric 2D ERSP grids)
INPUT_DIR = ANALYSIS_ROOT / r"01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY"

# Existing clustering outputs (unchanged for now)
CLUSTERING_OUTPUT_DIR  = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/clustering/kmeans"
FIGURES_EMBEDDINGS_DIR = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/figures/embeddings"
FIGURES_ERSP_DIR       = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/figures/ersp_clusters"
LOG_DIR                = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/logs"

# New: autoencoder-specific outputs
AE_ROOT        = ANALYSIS_ROOT / r"02_FBM_Clustering/outputs/autoencoder"
AE_MODEL_DIR   = AE_ROOT / "models"
AE_LATENT_DIR  = AE_ROOT / "latents"
AE_FIG_DIR     = AE_ROOT / "figures_recon"
AE_CONFIG_DIR  = AE_ROOT / "configs"

for d in [CLUSTERING_OUTPUT_DIR, FIGURES_EMBEDDINGS_DIR, FIGURES_ERSP_DIR,
          LOG_DIR, AE_ROOT, AE_MODEL_DIR, AE_LATENT_DIR, AE_FIG_DIR, AE_CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- WM reference maps (kept as in original script, not used here but harmless) ----
WHITE_MATTER_REFS_NUMS = {
    "PAT_3066": {"FIG":[6,5], "FOG":[15], "FOD":[12], "CAG":[11,10,9,8,7], "CAD":[8,6,5,3,2],
                 "IAG":[18,11,9,3,2], "IMG":[18,9,8,7,6,3], "AG":[3,2,1], "HAG":[1], "HPG":[7,2],
                 "PPG":[6,5,4,2], "TIG":[3], "AD":[8,6,4,3,2,1], "HAD":[3,2,1], "HPD":[2,1], "PPD":[6,5,4,3]},
    "PAT_3975": {"FOG":[15,10,8,3,1], "CPG":[10,6,5,4,3], "AG":[12], "HAG":[5,2,1], "PHG":[6,4],
                 "IAG":[8,6,4,3,2,1], "IMG":[13,12,11,6,3], "FOD":[14,13,12,11,9,6,5,3],
                 "CPD":[15,8,6,2], "AD":[7,4,3,2,1], "HAD":[3,2,1], "PHD":[11,7], "IAD":[10],
                 "IMD":[17,16,15,14,12,10,9,7]},
    "PAT_3415": {"IMG":[16,15,12,11,8,6,5,4,3,2], "IPG":[12,10,8,3,2,1], "HLG":[17,16,3,2]},
    "PAT_3965": {"ag":[3,4,5,6], "hpg":[8,9,10,11], "imd":[6,7,8,9,10,11,12]},
    "PAT_2868": {"POP":[3], "IDM":[4,1], "SMA":[2,1], "PPS":[3,2,1], "PRI":[3,2,1], "POM":[4,1],
                 "PPI":[2], "POI":[3,2,1]},
    "PAT_3455": {"AD":[2,1], "HAD":[2,1], "TOD":[6], "CPD":[12,11,10,9,6,5,2], "OPD":[5,4,3,2],
                 "PHD":[5], "OTD":[8], "TSP":[8,6,5,4], "IMD":[11,5,4,3], "IPD":[14,11]},
    "PAT_3390": {"IAG":[18,17,16,9,1], "CAG":[8,6,5,4,2], "CPG":[15,13,10,9,1], "HPG":[10,9,8],
                 "AG":[12,11], "HAG":[12,11,10,9], "FOG":[9,4,3,2,1]},
    "MicroEPI-B-01": {"pI_L":[8,9,10,11,12,13,14], "A_L":[4,5],
                      "ITG_L":[6,9],"sSMG_L":[2,3,5,6],
                      "IOG_L":[4,5,7],"LinG_L":[4,5]}
}

# ---- Data shape constants ----
N_TIME_FULL = 300
N_FREQ_FULL = 129

# Downsampled dimensions
N_FREQ_DS = 13
N_TIME_DS = 30

# ---- AE parameters ----
RANDOM_STATE = 42

LATENT_DIM   = 8
BATCH_SIZE   = 128
NUM_EPOCHS   = 50
LR           = 1e-3
ALPHA_WEIGHT = 1.0     # weight for positive values in loss
LAMBDA_L1    = 1e-4    # sparsity penalty on latent

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("SCRIPT_NAME:", SCRIPT_NAME)
print("ALGO_TAG   :", ALGO_TAG)
print("RUN_ID     :", RUN_ID)
print("Input dir  :", INPUT_DIR)
print("Using device:", DEVICE)

# ---- Seed everything for reproducibility ----
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(RANDOM_STATE)

## 1. Auto-detect patient IDs from outputs/04_ersp_LM_RAWONLY

PATIENT_IDS = sorted([d.name for d in INPUT_DIR.iterdir() if d.is_dir()])

print("Detected patient folders:")
for p in PATIENT_IDS:
    print("  -", p)
print("\nTotal patients detected:", len(PATIENT_IDS))


## 2. Helper: parse electrode name from filename (same logic as your clustering script)

def parse_electrode_from_filename(fname: str) -> str:
    """
    Example filenames:
        PAT_3301_picture_None_ERSP_AG2_TN.npy   -> electrode = 'AG2'
        EL035_reading_WM_ERSP_A_R10_TN.npy      -> electrode = 'A_R10'
        EL035_reading_WM_ERSP_Fp2_TN.npy        -> electrode = 'Fp2'
        EL030_audio_WM_ERSP_aH_L1_TN.npy        -> electrode = 'aH_L1'

    Assumes pattern: ..._ERSP_<electrode>_<suffix>.<ext>
    where <electrode> may contain underscores or dashes.
    """
    name = Path(fname).name
    if "_ERSP_" in name:
        _, right = name.split("_ERSP_", 1)
        right_no_ext = right.rsplit(".", 1)[0]  # remove extension
        parts = right_no_ext.split("_")

        if len(parts) == 1:
            electrode = right_no_ext
        else:
            electrode = "_".join(parts[:-1])  # last part assumed suffix
        return electrode
    else:
        return name.rsplit(".", 1)[0]


## 3. Helper: downsample ERSP to 13x30

def downsample_ersp_13x30(arr: np.ndarray) -> np.ndarray:
    """
    Downsample a (129, 300) array to (13, 30) by block averaging.
    - 12 freq bins of size 10, last freq bin of size 9.
    - 30 time bins of size 10.
    """
    if arr.shape != (N_FREQ_FULL, N_TIME_FULL):
        raise ValueError(f"Expected shape {(N_FREQ_FULL, N_TIME_FULL)}, got {arr.shape}")

    out = np.zeros((N_FREQ_DS, N_TIME_DS), dtype=np.float32)

    # Time bins: 30 bins of 10 samples each
    for t_bin in range(N_TIME_DS):
        t_start = t_bin * 10
        t_end   = t_start + 10

        for f_bin in range(N_FREQ_DS):
            if f_bin < N_FREQ_DS - 1:
                f_start = f_bin * 10
                f_end   = f_start + 10
            else:
                # last freq bin: 9 freqs (120..128)
                f_start = 12 * 10
                f_end   = N_FREQ_FULL  # 129

            block = arr[f_start:f_end, t_start:t_end]
            out[f_bin, t_bin] = block.mean()

    return out


## 4. Load ERSP_matrix .npy and build dataset

TASK = "LM"   # as before

ersp_ds_list = []
meta_rows = []

for pat in PATIENT_IDS:
    # Use ERSP_matrix (numeric ERSP grids)
    patient_dir = INPUT_DIR / pat / TASK / "ERSP_matrix"
    if not patient_dir.exists():
        print(f"[WARN] Missing ERSP_matrix folder for patient {pat}: {patient_dir}")
        continue

    condition_dirs = [d for d in patient_dir.iterdir() if d.is_dir()]
    if not condition_dirs:
        print(f"[WARN] No condition subfolders for {pat} in {patient_dir}")
        continue

    print(f"\nPatient {pat} — conditions found:", [d.name for d in condition_dirs])

    for cond_dir in condition_dirs:
        cond_name = cond_dir.name
        if cond_name not in ("audio", "picture", "reading"):
            continue

        npy_files = sorted(cond_dir.glob("*.npy"))
        if not npy_files:
            print(f"[WARN] No .npy files found in {cond_dir}")
            continue

        print(f"  Loading {len(npy_files)} .npy files from condition '{cond_name}'")

        for fpath in npy_files:
            try:
                arr = np.load(fpath)
                if arr.shape != (N_FREQ_FULL, N_TIME_FULL):
                    print(f"  [WARN] Incorrect shape {arr.shape} in {fpath}, skipping.")
                    continue

                img_ds = downsample_ersp_13x30(arr)  # (13, 30)
            except Exception as e:
                print(f"  [WARN] Failed to load/downsample {fpath}: {e}")
                continue

            ersp_ds_list.append(img_ds)

            meta_rows.append({
                "patient_id": pat,
                "condition": cond_name,
                "task": TASK,
                "electrode": parse_electrode_from_filename(fpath.name),
                "file_path": str(fpath),
            })

df_meta = pd.DataFrame(meta_rows)
df_meta.index.name = "sample_idx"
df_meta.reset_index(inplace=True)

print("\n=== Finished Loading ===")
print("Total ERSP samples loaded (downsampled):", len(ersp_ds_list))

if len(ersp_ds_list) == 0:
    raise RuntimeError("No ERSP_matrix .npy data found. Aborting.")

# Stack to (N, 13, 30)
X_raw = np.stack(ersp_ds_list, axis=0)  # shape (N, 13, 30)
print("X_raw shape:", X_raw.shape)


## 5. Normalize (z-score) across dataset per pixel

N_SAMPLES = X_raw.shape[0]
X_flat = X_raw.reshape(N_SAMPLES, -1)  # (N, 390)

mean_vec = X_flat.mean(axis=0)
std_vec  = X_flat.std(axis=0) + 1e-8

X_norm_flat = (X_flat - mean_vec) / std_vec
X_norm = X_norm_flat.reshape(N_SAMPLES, 1, N_FREQ_DS, N_TIME_DS).astype(np.float32)

print("X_norm shape:", X_norm.shape)

# Save normalization stats for future use
norm_mean_path = AE_ROOT / f"norm_mean_{RUN_ID}.npy"
norm_std_path  = AE_ROOT / f"norm_std_{RUN_ID}.npy"
np.save(norm_mean_path, mean_vec)
np.save(norm_std_path, std_vec)
print(f"Saved normalization mean/std to:\n  {norm_mean_path}\n  {norm_std_path}")


## 6. PyTorch Dataset & DataLoader

class ERSPDataset(Dataset):
    def __init__(self, X: np.ndarray):
        # X: (N, 1, 13, 30)
        self.X = torch.from_numpy(X)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx]


# Train/val split
train_idx, val_idx = train_test_split(
    np.arange(N_SAMPLES),
    test_size=0.1,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=None  # could stratify by patient or condition if desired
)

X_train = X_norm[train_idx]
X_val   = X_norm[val_idx]

train_ds = ERSPDataset(X_train)
val_ds   = ERSPDataset(X_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")


## 7. Conv Sparse Autoencoder

class ConvSparseAE(nn.Module):
    def __init__(self, latent_dim: int):
        super().__init__()
        # Encoder
        self.enc_conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.enc_pool1 = nn.MaxPool2d(kernel_size=2, stride=2)   # (13,30) -> (6,15)
        self.enc_conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.enc_pool2 = nn.MaxPool2d(kernel_size=(2, 3), stride=(2, 3))  # (6,15) -> (3,5)

        self.flatten = nn.Flatten()
        self.fc_enc  = nn.Linear(32 * 3 * 5, latent_dim)

        # Decoder
        self.fc_dec  = nn.Linear(latent_dim, 32 * 3 * 5)
        self.dec_upsample1 = nn.Upsample(scale_factor=(2, 3), mode="nearest")  # (3,5)->(6,15)
        self.dec_conv1     = nn.Conv2d(32, 16, kernel_size=3, padding=1)
        self.dec_upsample2 = nn.Upsample(scale_factor=(2, 2), mode="nearest")  # (6,15)->(12,30)
        self.dec_conv2     = nn.Conv2d(16, 1, kernel_size=3, padding=1)

        self.act = nn.ReLU()

    def encode(self, x):
        # x: (B,1,13,30)
        x = self.act(self.enc_conv1(x))
        x = self.enc_pool1(x)          # (B,16,6,15)
        x = self.act(self.enc_conv2(x))
        x = self.enc_pool2(x)          # (B,32,3,5)
        x = self.flatten(x)            # (B,480)
        z = self.fc_enc(x)             # (B,latent_dim)
        z = self.act(z)                # ReLU latent
        return z

    def decode(self, z):
        # z: (B,latent_dim)
        x = self.fc_dec(z)                     # (B,480)
        x = x.view(-1, 32, 3, 5)               # (B,32,3,5)
        x = self.dec_upsample1(x)              # (B,32,6,15)
        x = self.act(self.dec_conv1(x))        # (B,16,6,15)
        x = self.dec_upsample2(x)              # (B,16,12,30)
        # Need (13,30): pad one freq row at bottom
        if x.shape[2] == 12:
            pad = (0, 0, 0, 1)  # (left,right,top,bottom) in (W,H)
            x = nn.functional.pad(x, pad, mode="replicate")  # (B,16,13,30)
        x = self.dec_conv2(x)                  # (B,1,13,30)
        return x

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z


model = ConvSparseAE(latent_dim=LATENT_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(model)


## 8. Weighted loss function (emphasizing positive values)

def weighted_mse_loss(x, x_hat, alpha: float):
    """
    x, x_hat: (B,1,13,30)
    weight = 1 + alpha * relu(x)
    """
    pos = torch.relu(x)
    w = 1.0 + alpha * pos
    diff2 = (x - x_hat) ** 2
    loss = (w * diff2).sum() / w.sum()
    return loss


## 9. Training loop

best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_losses = []

    for batch in train_loader:
        batch = batch.to(DEVICE)  # (B,1,13,30)

        optimizer.zero_grad()
        x_hat, z = model(batch)

        recon_loss = weighted_mse_loss(batch, x_hat, ALPHA_WEIGHT)
        sparse_loss = LAMBDA_L1 * torch.mean(torch.abs(z))
        loss = recon_loss + sparse_loss

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            x_hat, z = model(batch)
            recon_loss = weighted_mse_loss(batch, x_hat, ALPHA_WEIGHT)
            sparse_loss = LAMBDA_L1 * torch.mean(torch.abs(z))
            loss = recon_loss + sparse_loss
            val_losses.append(loss.item())

    train_loss = float(np.mean(train_losses))
    val_loss   = float(np.mean(val_losses))
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"[Epoch {epoch:03d}] train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_path = AE_MODEL_DIR / f"ae_matrix_{RUN_ID}_best.pt"
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> New best model saved to {best_model_path}")

# Save final model
final_model_path = AE_MODEL_DIR / f"ae_matrix_{RUN_ID}_final.pt"
torch.save(model.state_dict(), final_model_path)
print(f"Final model saved to {final_model_path}")

# Save training history
history_path = AE_CONFIG_DIR / f"history_{RUN_ID}.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"Training history saved to {history_path}")


## 10. Extract latent embeddings for all samples & save

model.eval()

X_tensor = torch.from_numpy(X_norm).to(DEVICE)
all_latents = []

with torch.no_grad():
    for i in range(0, N_SAMPLES, BATCH_SIZE):
        batch = X_tensor[i:i+BATCH_SIZE]
        z = model.encode(batch)
        all_latents.append(z.cpu().numpy())

all_latents = np.vstack(all_latents)  # (N_SAMPLES, LATENT_DIM)
print("Latent shape:", all_latents.shape)

latents_path = AE_LATENT_DIR / f"latents_{RUN_ID}.npy"
meta_path    = AE_LATENT_DIR / f"latents_meta_{RUN_ID}.csv"

np.save(latents_path, all_latents)
df_meta.to_csv(meta_path, index=False)

print(f"Saved latents to: {latents_path}")
print(f"Saved meta to:    {meta_path}")


## 11. Save a few example reconstructions for sanity check

num_examples = min(16, N_SAMPLES)
example_indices = np.random.choice(N_SAMPLES, size=num_examples, replace=False)

X_ex = X_norm[example_indices]
X_ex_tensor = torch.from_numpy(X_ex).to(DEVICE)

with torch.no_grad():
    X_rec_tensor, _ = model(X_ex_tensor)

X_ex_np  = X_ex_tensor.cpu().numpy()   # (N_ex,1,13,30)
X_rec_np = X_rec_tensor.cpu().numpy()

fig, axes = plt.subplots(num_examples, 2, figsize=(6, 2 * num_examples))

for i in range(num_examples):
    ax_org = axes[i, 0]
    ax_rec = axes[i, 1]

    ax_org.imshow(X_ex_np[i, 0], aspect="auto", origin="lower")
    ax_org.set_title(f"Orig {example_indices[i]}")
    ax_org.axis("off")

    ax_rec.imshow(X_rec_np[i, 0], aspect="auto", origin="lower")
    ax_rec.set_title(f"Recon {example_indices[i]}")
    ax_rec.axis("off")

plt.tight_layout()
fig_path = AE_FIG_DIR / f"recon_examples_{RUN_ID}.png"
plt.savefig(fig_path, dpi=150)
plt.close(fig)

print(f"Example reconstructions saved to {fig_path}")


## 12. Save config for reproducibility

config = {
    "SCRIPT_NAME": SCRIPT_NAME,
    "ALGO_TAG": ALGO_TAG,
    "RUN_ID": RUN_ID,
    "INPUT_DIR": str(INPUT_DIR),
    "AE_ROOT": str(AE_ROOT),
    "LATENT_DIM": LATENT_DIM,
    "BATCH_SIZE": BATCH_SIZE,
    "NUM_EPOCHS": NUM_EPOCHS,
    "LR": LR,
    "ALPHA_WEIGHT": ALPHA_WEIGHT,
    "LAMBDA_L1": LAMBDA_L1,
    "N_FREQ_FULL": N_FREQ_FULL,
    "N_TIME_FULL": N_TIME_FULL,
    "N_FREQ_DS": N_FREQ_DS,
    "N_TIME_DS": N_TIME_DS,
    "DEVICE": str(DEVICE),
    "N_SAMPLES": int(N_SAMPLES),
}

config_path = AE_CONFIG_DIR / f"config_{RUN_ID}.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Config saved to {config_path}")


SCRIPT_NAME: 220_autoencoder_ersp_matrix.py
ALGO_TAG   : autoencoder_clean
RUN_ID     : 20251201_181016
Input dir  : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY
Using device: cpu
Detected patient folders:
  - EL030
  - EL035
  - EL036
  - EL037
  - EL038
  - EL040
  - EL042
  - PAT_2868
  - PAT_3066
  - PAT_3301
  - PAT_3390
  - PAT_3415
  - PAT_3455
  - PAT_3780
  - PAT_3965
  - PAT_3975

Total patients detected: 16

Patient EL030 — conditions found: ['audio', 'picture', 'reading']
  Loading 89 .npy files from condition 'audio'
  Loading 89 .npy files from condition 'picture'
  Loading 89 .npy files from condition 'reading'

Patient EL035 — conditions found: ['audio', 'picture', 'reading']
  Loading 118 .npy files from condition 'audio'
  Loading 118 .npy files from condition 'picture'
  Loading 118 .npy files from condition 'reading'

Patient EL036 — conditions found: ['audio', 'picture', 'reading']
  Loading 63 .npy file